## Evaluate Presidio Analyzer using the Presidio Evaluator framework

This notebook demonstrates how to evaluate a Presidio instance using the presidio-evaluator framework
Steps:
1. Load dataset from file
2. Simple dataset statistics
3. Define the AnalyzerEngine object (and its parameters)
4. Align the dataset's entities to Presidio's entities
5. Set up the Evaluator object
6. Run experiment
7. Evaluate results
8. Error analysis

For an example with a custom Presidio instance, see [notebook 5](5_Evaluate_Custom_Presidio_Analyzer.ipynb).

In [1]:
# install presidio evaluator via pip if not yet installed

#!pip install presidio-evaluator

In [2]:
from pathlib import Path
from pprint import pprint
from collections import Counter
from typing import Dict, List
import json

from presidio_evaluator import InputSample
from presidio_evaluator.evaluation import SpanEvaluator, ModelError, Plotter
from presidio_evaluator.experiment_tracking import get_experiment_tracker

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

%reload_ext autoreload
%autoreload 2

/Users/jyu36/code/ic-llm/presidio/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/Users/jyu36/code/ic-llm/presidio/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


stanza and spacy_stanza are not installed
Flair is not installed by default


## 1. Load dataset from file

In [3]:
dataset_name = "synth_dataset_v2.json"
dataset = InputSample.read_dataset_json(Path(Path.cwd(), "data", dataset_name))
print(len(dataset))

tokenizing input:   0%|          | 0/1497 [00:00<?, ?it/s]

loading model en_core_web_sm


tokenizing input: 100%|██████████| 1497/1497 [00:06<00:00, 232.64it/s]

1497


This dataset was auto generated. See more info here [Synthetic data generation](1_Generate_data.ipynb).

In [4]:
def get_entity_counts(dataset: List[InputSample]) -> Dict:
    """Return a dictionary with counter per entity type."""
    entity_counter = Counter()
    for sample in dataset:
        for tag in sample.tags:
            entity_counter[tag] += 1
    return entity_counter


In [5]:
def get_entity_counts(dataset: List[InputSample]) -> Dict:
    """Return a dictionary with counter per entity type."""
    entity_counter = Counter()
    for sample in dataset:
        print("SAMPLE:", sample)
        for tag in sample.tags:
            print("TAG:", tag)
            entity_counter[tag] += 1
    return entity_counter

entity_counts = get_entity_counts(dataset)
print(entity_counts)
pprint(entity_counts.most_common(), compact=True)

SAMPLE: Full text: The Exversion Orchestra was founded in 1977. Since then, it has grown from a volunteer community orchestra to a fully professional orchestra serving Southern Tunisia
Spans: [Span(type: GPE, value: Tunisia, char_span: [158: 165]), Span(type: DATE_TIME, value: 1977, char_span: [39: 43]), Span(type: ORGANIZATION, value: Exversion, char_span: [4: 13])]

TAG: O
TAG: ORGANIZATION
TAG: O
TAG: O
TAG: O
TAG: O
TAG: DATE_TIME
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: GPE
SAMPLE: Full text: My name is Rubija
Spans: [Span(type: PERSON, value: Rubija, char_span: [11: 17])]

TAG: O
TAG: O
TAG: O
TAG: PERSON
SAMPLE: Full text: What is the limit for card 4454794511390933?
Spans: [Span(type: CREDIT_CARD, value: 4454794511390933, char_span: [27: 43])]

TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: CREDIT_CARD
TAG: O
SAMPLE: Full text: When they weren't singing about Hobbits, satanic feline

## 2. Simple dataset statistics

In [6]:
entity_counts = get_entity_counts(dataset)
print("Count per entity:")
pprint(entity_counts.most_common(), compact=True)

print("\nMin and max number of tokens in dataset: "\
f"Min: {min([len(sample.tokens) for sample in dataset])}, "\
f"Max: {max([len(sample.tokens) for sample in dataset])}")

print(f"Min and max sentence length in dataset: " \
f"Min: {min([len(sample.full_text) for sample in dataset])}, "\
f"Max: {max([len(sample.full_text) for sample in dataset])}")

print("\nExample InputSample:")
print(dataset[0])

SAMPLE: Full text: The Exversion Orchestra was founded in 1977. Since then, it has grown from a volunteer community orchestra to a fully professional orchestra serving Southern Tunisia
Spans: [Span(type: GPE, value: Tunisia, char_span: [158: 165]), Span(type: DATE_TIME, value: 1977, char_span: [39: 43]), Span(type: ORGANIZATION, value: Exversion, char_span: [4: 13])]

TAG: O
TAG: ORGANIZATION
TAG: O
TAG: O
TAG: O
TAG: O
TAG: DATE_TIME
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: GPE
SAMPLE: Full text: My name is Rubija
Spans: [Span(type: PERSON, value: Rubija, char_span: [11: 17])]

TAG: O
TAG: O
TAG: O
TAG: PERSON
SAMPLE: Full text: What is the limit for card 4454794511390933?
Spans: [Span(type: CREDIT_CARD, value: 4454794511390933, char_span: [27: 43])]

TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: CREDIT_CARD
TAG: O
SAMPLE: Full text: When they weren't singing about Hobbits, satanic feline

In [7]:
print("A few examples sentences containing each entity:\n")
for entity in entity_counts.keys():
    samples = [sample for sample in dataset if entity in set(sample.tags)]
    if len(samples) > 1 and entity != "O":
        print(f"Entity: <{entity}> two example sentences:\n"
              f"\n1) {samples[0].full_text}"
              f"\n2) {samples[1].full_text}"
              f"\n------------------------------------\n")

A few examples sentences containing each entity:

Entity: <ORGANIZATION> two example sentences:

1) The Exversion Orchestra was founded in 1977. Since then, it has grown from a volunteer community orchestra to a fully professional orchestra serving Southern Tunisia
2) Microbilt Corporation is the brainchild of our 3 founders: Kónya, Becker and Vasquez.  The idea was born (on the beach) while they were constructing a website to be the basis of another start-up idea.
------------------------------------

Entity: <DATE_TIME> two example sentences:

1) The Exversion Orchestra was founded in 1977. Since then, it has grown from a volunteer community orchestra to a fully professional orchestra serving Southern Tunisia
2) When: 2000-04-16 11:34:35
Where: UPTON SCUDAMORE Country Club.
------------------------------------

Entity: <GPE> two example sentences:

1) The Exversion Orchestra was founded in 1977. Since then, it has grown from a volunteer community orchestra to a fully professional orc

## 3. Define the AnalyzerEngine object 
Using Presidio with default parameters (not recommended, it's used here for simplicity). For an example on customization, see [notebook 5](5_Evaluate_Custom_Presidio_Analyzer.ipynb)

In [8]:
from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider

# Loading the vanilla Analyzer Engine, with the default NER model.
configuration = {
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "en", "model_name": "en_core_web_sm"}],
}
provider = NlpEngineProvider(nlp_configuration=configuration)
nlp_engine_with_sm = provider.create_engine()
analyzer_engine = AnalyzerEngine(nlp_engine=nlp_engine_with_sm, default_score_threshold=0.4)

pprint(f"Supported entities for English:")
pprint(analyzer_engine.get_supported_entities("en"), compact=True)

print(f"\nLoaded recognizers for English:")
pprint([rec.name for rec in analyzer_engine.registry.get_recognizers("en", all_fields=True)], compact=True)

print(f"\nLoaded NER models:")
pprint(analyzer_engine.nlp_engine.models)

'Supported entities for English:'
['NRP', 'UK_NHS', 'AGE', 'US_BANK_NUMBER', 'CRYPTO', 'US_PASSPORT',
 'ORGANIZATION', 'URL', 'EMAIL', 'LOCATION', 'ID', 'MEDICAL_LICENSE',
 'IBAN_CODE', 'IP_ADDRESS', 'PHONE_NUMBER', 'PERSON', 'US_DRIVER_LICENSE',
 'DATE_TIME', 'CREDIT_CARD', 'MAC_ADDRESS', 'US_ITIN', 'US_SSN',
 'EMAIL_ADDRESS']

Loaded recognizers for English:
['CreditCardRecognizer', 'UsBankRecognizer', 'UsLicenseRecognizer',
 'UsItinRecognizer', 'UsPassportRecognizer', 'UsSsnRecognizer', 'NhsRecognizer',
 'CryptoRecognizer', 'DateRecognizer', 'EmailRecognizer', 'IbanRecognizer',
 'IpRecognizer', 'MedicalLicenseRecognizer', 'MacAddressRecognizer',
 'PhoneRecognizer', 'UrlRecognizer', 'SpacyRecognizer']

Loaded NER models:
[{'lang_code': 'en', 'model_name': 'en_core_web_sm'}]


## 4. Align the dataset's entities to Presidio's entities

There is possibly a difference between the names of entities in the dataset, and the names of entities Presidio can detect.
For example, it could be that a dataset labels a name as PER while Presidio returns PERSON. To be able to compare the predicted value to the actual and gather metrics, an alignment between the entity names is necessary. Consider changing the mapping if your dataset and/or Presidio instance supports difference entity types.

In [9]:
from presidio_evaluator.models import  PresidioAnalyzerWrapper

entities_mapping=PresidioAnalyzerWrapper.presidio_entities_map # default mapping

print("Using this mapping between the dataset and Presidio's entities:")
pprint(entities_mapping, compact=True)


dataset = SpanEvaluator.align_entity_types(
    dataset, 
    entities_mapping=entities_mapping, 
    allow_missing_mappings=True
)
new_entity_counts = get_entity_counts(dataset)
print("\nCount per entity after alignment:")
pprint(new_entity_counts.most_common(), compact=True)

dataset_entities = list(new_entity_counts.values())


Using this mapping between the dataset and Presidio's entities:
{'ADDRESS': 'LOCATION',
 'AGE': 'AGE',
 'BIRTHDAY': 'DATE_TIME',
 'CITY': 'LOCATION',
 'CREDIT_CARD': 'CREDIT_CARD',
 'CREDIT_CARD_NUMBER': 'CREDIT_CARD',
 'DATE': 'DATE_TIME',
 'DATE_OF_BIRTH': 'DATE_TIME',
 'DATE_TIME': 'DATE_TIME',
 'DOB': 'DATE_TIME',
 'DOMAIN': 'URL',
 'DOMAIN_NAME': 'URL',
 'EMAIL': 'EMAIL_ADDRESS',
 'EMAIL_ADDRESS': 'EMAIL_ADDRESS',
 'FACILITY': 'LOCATION',
 'FIRST_NAME': 'PERSON',
 'GPE': 'LOCATION',
 'HCW': 'PERSON',
 'HOSP': 'ORGANIZATION',
 'HOSPITAL': 'ORGANIZATION',
 'IBAN': 'IBAN_CODE',
 'IBAN_CODE': 'IBAN_CODE',
 'ID': 'ID',
 'IP_ADDRESS': 'IP_ADDRESS',
 'LAST_NAME': 'PERSON',
 'LOC': 'LOCATION',
 'LOCATION': 'LOCATION',
 'NAME': 'PERSON',
 'NATIONALITY': 'NRP',
 'NORP': 'NRP',
 'NRP': 'NRP',
 'O': 'O',
 'ORG': 'ORGANIZATION',
 'ORGANIZATION': 'ORGANIZATION',
 'PATIENT': 'PERSON',
 'PATORG': 'ORGANIZATION',
 'PER': 'PERSON',
 'PERSON': 'PERSON',
 'PHONE': 'PHONE_NUMBER',
 'PHONE_NUMBER': 'PH

## 5. Set up the Evaluator object

In [10]:
# Set up the experiment tracker to log the experiment for reproducibility
experiment = get_experiment_tracker()

# Create the evaluator object
evaluator = SpanEvaluator(model=analyzer_engine)


# Track model and dataset params
params = {"dataset_name": dataset_name, 
          "model_name": evaluator.model.name}
params.update(evaluator.model.to_log())
experiment.log_parameters(params)
experiment.log_dataset_hash(dataset)
experiment.log_parameter("entity_mappings", json.dumps(entities_mapping))

--------
Entities supported by this Presidio Analyzer instance:
NRP, UK_NHS, AGE, US_BANK_NUMBER, CRYPTO, US_PASSPORT, ORGANIZATION, URL, EMAIL, LOCATION, ID, MEDICAL_LICENSE, IBAN_CODE, IP_ADDRESS, PHONE_NUMBER, PERSON, US_DRIVER_LICENSE, DATE_TIME, CREDIT_CARD, MAC_ADDRESS, US_ITIN, US_SSN, EMAIL_ADDRESS


/Users/jyu36/code/ic-llm/presidio/.venv/lib/python3.12/site-packages/presidio_evaluator/evaluation/base_evaluator.py:83: UserWarning: skip words not provided, using default skip words. If you want the evaluation to not use skip words, pass skip_words=[]
  warnings.warn("skip words not provided, using default skip words. "


## 6. Run experiment

In [11]:
%%time

## Run experiment

evaluation_results = evaluator.evaluate_all(dataset)
results = evaluator.calculate_score(evaluation_results)

# Track experiment results
experiment.log_metrics(results.to_log())
entities, confmatrix = results.to_confusion_matrix()
experiment.log_confusion_matrix(matrix=confmatrix, 
                                labels=entities)

# end experiment
experiment.end()

Running model PresidioAnalyzerWrapper on dataset...
Finished running model on dataset
saving experiment data to /Users/jyu36/code/ic-llm/presidio/experiment_20260320-214042.json
CPU times: user 9.08 s, sys: 114 ms, total: 9.2 s
Wall time: 9.37 s


## 7. Evaluate results

In [12]:
# Plot output
plotter = Plotter(results=results, 
                  model_name = evaluator.model.name, 
                  save_as="svg",
                  beta = 2) 

# Path of the directory to save the plots
output_folder = Path(Path.cwd(), "plotter_output")
plotter.plot_scores(output_folder=output_folder)

In [13]:
pprint({"PII F":results.pii_f, "PII recall": results.pii_recall, "PII precision": results.pii_precision})

{'PII F': 0.6532222709338009,
 'PII precision': 0.6903474903474903,
 'PII recall': 0.6445565969718817}


### 7a. False positives
#### Most common false positive tokens:

In [14]:
ModelError.most_common_fp_tokens(results.model_errors)

Most common false positive tokens:
[('iban', 15),
 ('comedy kids family mystery suspense', 13),
 ('ssn', 10),
 ('couple', 10),
 ('greek', 9),
 ('8', 9),
 ('salesperson', 9),
 ('sunday', 9),
 ('fuse tv', 8),
 ('60s', 8)]
---------------
Example sentence with each FP token:
	- IBAN (`iban` pred as ORGANIZATION)
	- Comedy , Kids & Family Mystery & Suspense (`comedy kids family mystery suspense` pred as ORGANIZATION)
	- SSN (`ssn` pred as ORGANIZATION)
	- a couple of months (`couple` pred as DATE_TIME)
	- Greek (`greek` pred as O)
	- 8 + years (`8` pred as DATE_TIME)
	- Salesperson (`salesperson` pred as PERSON)
	- the last Sunday (`sunday` pred as DATE_TIME)
	- Fuse TV (`fuse tv` pred as ORGANIZATION)
	- the 60s (`60s` pred as DATE_TIME)


[('iban', 15),
 ('comedy kids family mystery suspense', 13),
 ('ssn', 10),
 ('couple', 10),
 ('greek', 9),
 ('8', 9),
 ('salesperson', 9),
 ('sunday', 9),
 ('fuse tv', 8),
 ('60s', 8)]

#### More FP analysis

In [15]:
fps_df = ModelError.get_fps_dataframe(results.model_errors, entity=["PERSON"])
fps_df[["full_text", "token", "annotation", "prediction"]].head(20)

,full_text,token,annotation,prediction
0,Salesperson,salesperson,O,PERSON
1,Discipline,discipline,O,PERSON
2,Orchestra,orchestra,O,PERSON
3,Discipline,discipline,O,PERSON
4,Salesperson,salesperson,O,PERSON
5,Salesperson,salesperson,O,PERSON
6,Discipline,discipline,O,PERSON
7,salesperson,salesperson,O,PERSON
8,Salesperson,salesperson,O,PERSON
9,Salesperson,salesperson,O,PERSON


### 7b. False negatives (FN)

#### Most common false negative examples + a few samples with FN

In [16]:
ModelError.most_common_fn_tokens(results.model_errors, n=15)

Most common false negative tokens:
[('greenlander', 11),
 ('dr.', 9),
 ('ms.', 9),
 ('46', 6),
 ('61', 6),
 ('64', 6),
 ('65', 6),
 ('54', 6),
 ('espoo', 6),
 ('21', 4),
 ('78', 4),
 ('wisława szczepańska', 4),
 ('47', 4),
 ('63', 4),
 ('58', 4)]
---------------
Example sentence with each FN token:
	- Greenlander (`greenlander` annotated as O)
	- Dr. (`dr.` annotated as O)
	- Ms. (`ms.` annotated as O)
	- 46 (`46` annotated as O)
	- 61 (`61` annotated as O)
	- 64 (`64` annotated as O)
	- 65 (`65` annotated as O)
	- 54 (`54` annotated as O)
	- ESPOO (`espoo` annotated as O)
	- 21 (`21` annotated as O)
	- 78 (`78` annotated as O)
	- Wisława Szczepańska (`wisława szczepańska` annotated as O)
	- 47 (`47` annotated as O)
	- 63 (`63` annotated as O)
	- 58 (`58` annotated as O)


[('greenlander', 11),
 ('dr.', 9),
 ('ms.', 9),
 ('46', 6),
 ('61', 6),
 ('64', 6),
 ('65', 6),
 ('54', 6),
 ('espoo', 6),
 ('21', 4),
 ('78', 4),
 ('wisława szczepańska', 4),
 ('47', 4),
 ('63', 4),
 ('58', 4)]

#### More FN analysis

In [17]:
fns_df = ModelError.get_fns_dataframe(results.model_errors)

In [18]:
fns_df[["full_text", "token", "annotation", "prediction"]].head(20)

,full_text,token,annotation,prediction
0,Exversion,exversion,O,ORGANIZATION
1,Tunisia,tunisia,O,LOCATION
2,Rubija,rubija,O,PERSON
3,Faina D. Yefremova Yefremova,faina d. yefremova yefremova,O,PERSON
4,Johannes Varvio,johannes varvio,O,PERSON
5,2270 - 66 - 1551,2270 66 1551,O,US_DRIVER_LICENSE
6,79,79,O,AGE
7,28245 Puruntie 82 Apt . 595 LAPPEENRANTA SK,28245 puruntie 82 595 lappeenranta sk,O,LOCATION
8,53650,53650,O,ZIP_CODE
9,Ubul Nicole Mary John,ubul nicole mary john,O,PERSON
